Step One: Process and merge Fake News Dataset and CoAID Dataset

In [3]:
import pandas as pd
import os
import glob

# Process fake news dataset: only keep columns title and label
df_fakenews = pd.read_csv('news.csv')
df_fakenews_clean = df_fakenews[['title', 'label']].copy()

# Map labels to binary values (0 for REAL, 1 for FAKE)
df_fakenews_clean['label'] = df_fakenews_clean['label'].map({'REAL': 0, 'FAKE': 1})

# Rename columns to match standard format
df_fakenews_clean = df_fakenews_clean.rename(columns={
    'title': 'text', 
    'label': 'label'
})

# Process coAID datraset: only keep title column and create a new label column 
coaid_folder_path = 'ClaimFakeCOVID-19_5.csv' 

# Find all CSV files in that folder
coaid_files = glob.glob(coaid_folder_path)
coaid_dataframes = []

for file_path in coaid_files:
    df_temp = pd.read_csv(file_path)
    
    # Isolate the title and extract file name
    df_clean = df_temp[['title']].copy()
    file_name = os.path.basename(file_path)
    
    # Check the filename for 'Real' or 'Fake' and assign the label
    if 'Real' in file_name:
        df_clean['label'] = 0
    elif 'Fake' in file_name:
        df_clean['label'] = 1
    else:
        print(f"Warning: Could not determine label for {file_name}")
        continue # Skip if name doesn't match format
        
    df_clean = df_clean.rename(columns={'title': 'text'})
    coaid_dataframes.append(df_clean)

# Combine all individual CoAID dataframes
df_coaid_master = pd.concat(coaid_dataframes, ignore_index=True)

# 4. Merge and shuffle both datasets
combined_df = pd.concat([df_fakenews_clean, df_coaid_master], ignore_index=True)
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Verify the final format
print(combined_df.head())

                                                text  label
0                      How Hillary Clinton could win      0
1  Abby Martin Exposes Hillary Clinton Chair John...      1
2           Meet Ted Cruz's top fundraiser: his wife      0
3  Trump’s impending nomination means it’s time f...      0
4  Without Bold Agenda, Warn Progressives, A Clin...      1


Step Two: Clean data in 'text' column by removing stopwords, special characters, and hashtags. Covert all letters to lowercase, and enforce 50 word limit

In [5]:
import re
import torch
import nltk
from nltk.corpus import stopwords
from torch.utils.data import Dataset, DataLoader
from collections import Counter

# Download the standard stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sissi\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [6]:
def clean_and_limit_text(text):
    # Convert to string, then lowercase
    text = str(text).lower()
    
    # Remove hashtags and URLs
    text = re.sub(r'#\w+', '', text) 
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Remove special characters
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Tokenize by splitting into words and remove stopwords
    words = text.split()
    cleaned_words = [word for word in words if word not in stop_words]
    
    # Enforce 50 word limit
    cleaned_words = cleaned_words[:50]
    
    return cleaned_words

combined_df['clean_tokens'] = combined_df['text'].apply(clean_and_limit_text)

In [7]:
print(combined_df.head())

                                                text  label  \
0                      How Hillary Clinton could win      0   
1  Abby Martin Exposes Hillary Clinton Chair John...      1   
2           Meet Ted Cruz's top fundraiser: his wife      0   
3  Trump’s impending nomination means it’s time f...      0   
4  Without Bold Agenda, Warn Progressives, A Clin...      1   

                                        clean_tokens  
0                     [hillary, clinton, could, win]  
1  [abby, martin, exposes, hillary, clinton, chai...  
2          [meet, ted, cruzs, top, fundraiser, wife]  
3  [trumps, impending, nomination, means, time, t...  
4  [without, bold, agenda, warn, progressives, cl...  


Step Three: Calculate and record statistics

In [8]:
# Calculate real/fake claim counts
class_counts = combined_df['label'].value_counts()

print("Cleaned Data Statistics: ")
print(f"Total samples: {len(combined_df)}")
print(f"Real Claims (0): {class_counts.get(0, 0)}")
print(f"Fake Claims (1): {class_counts.get(1, 0)}")

print("\nExample Cleaned Sample: ")
print(f"Original: {combined_df.iloc[0]['text']}")
print(f"Cleaned & Tokenized: {combined_df.iloc[0]['clean_tokens']}")
print(f"Label: {combined_df.iloc[0]['label']}")

Cleaned Data Statistics: 
Total samples: 6362
Real Claims (0): 3171
Fake Claims (1): 3191

Example Cleaned Sample: 
Original: How Hillary Clinton could win
Cleaned & Tokenized: ['hillary', 'clinton', 'could', 'win']
Label: 0


Step Four: Convert to numerical tensors

In [9]:
# Create a vocabulary for all tokens in the dataset
all_words = [word for tokens in combined_df['clean_tokens'] for word in tokens]
vocab_counts = Counter(all_words)

# Map each word to a integer. Start at 1 to save 0 for padding 
vocab_to_int = {word: i+1 for i, (word, count) in enumerate(vocab_counts.items())}

def text_to_ints(tokens):
    return [vocab_to_int[word] for word in tokens]

combined_df['numerical_tokens'] = combined_df['clean_tokens'].apply(text_to_ints)

Step Five: Pad the sequences

In [10]:
SEQ_LENGTH = 50

def pad_features(numerical_tokens, seq_length):
    # Create an array of zeros and get length of token list
    features = torch.zeros(seq_length, dtype=torch.int64)
    token_len = len(numerical_tokens)
    
    # If the sentence is empty after cleaning, return zeros
    if token_len == 0:
        return features
        
    # Place the tokens into the tensor
    features[:token_len] = torch.tensor(numerical_tokens)
    return features

# Apply padding
padded_tensors = torch.stack(
    combined_df['numerical_tokens'].apply(lambda x: pad_features(x, SEQ_LENGTH)).tolist()
)

# Extract labels as a tensor
labels_tensor = torch.tensor(combined_df['label'].values, dtype=torch.float32)

Step Six: Wrap in Pytorch Dataset and DataLoader

In [ ]:
class MedicalMisinfoDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

full_dataset = MedicalMisinfoDataset(padded_tensors, labels_tensor)

# Wrap it in a DataLoader for batching (32 samples per batch)
BATCH_SIZE = 32
train_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Test the loader
dataiter = iter(train_loader)
sample_x, sample_y = next(dataiter)

print("\n--- DataLoader Verification ---")
print(f"Batch X shape: {sample_x.shape}")
print(f"Batch Y shape: {sample_y.shape}")


--- DataLoader Verification ---
Batch X shape: torch.Size([32, 50])
Batch Y shape: torch.Size([32])


Step Seven: Save the cleaned and processed dataset


In [14]:
# Save to dataset_csv folder without the index column (better readability)
csv_folder = 'dataset\\dataset_csv'
csv_file_path = os.path.join(csv_folder, "cleaned_medical_claims.csv")
combined_df.to_csv(csv_file_path, index=False)
print(f"Dataframe successfully saved to: {csv_file_path}")

# Save PyTorch Tensors for later use
tensor_folder = 'dataset\\dataset_tensors'
tensor_file_path = os.path.join(tensor_folder, "medical_tensors.pt")

torch.save({
    'features': padded_tensors,
    'labels': labels_tensor
}, tensor_file_path)
print(f"PyTorch tensors successfully saved to: {tensor_file_path}")

Dataframe successfully saved to: dataset\dataset_csv\cleaned_medical_claims.csv
PyTorch tensors successfully saved to: dataset\dataset_tensors\medical_tensors.pt
